# ECE452: Vision Transformer Inference and Analysis
**Comparison of Orthogonal vs. Xavier Initialization**

This notebook loads the trained Vision Transformer models from our experiments to perform inference on the MNIST test set. Our goal is to visually inspect the predictions and extract the internal representations (e.g., attention weights) to support our gradient flow hypothesis.

## 1. Environment Setup
Loading the required libraries for model architecture, data loading, and visualization.

In [2]:
import Pkg
# Tell Julia to use the environment in the parent directory (where Project.toml is)
Pkg.activate("..") 
Pkg.add("NNlib") # Ensures NNlib is explicitly available
Pkg.instantiate()

# Now load the libraries
using Flux
using NNlib
using JLD2
using MLDatasets
using Plots
using Random
using LinearAlgebra

# Define the paths
const RESULTS_DIR = joinpath("..", "results")
const LOGS_DIR    = joinpath(RESULTS_DIR, "logs")

# ViT Hyperparameters (MUST match training)
const IMG_SIZE   = 28
const PATCH_SIZE = 7
const IN_CHANNELS = 1
const EMBED_DIM  = 64
const NUM_HEADS  = 4
const MLP_RATIO  = 2
const NUM_BLOCKS = 2
const NUM_CLASSES = 10

println("Environment activated and libraries loaded ✓")

  Activating project at `c:\Users\nickb\Documents\vit-gradient-flow`
   Resolving package versions...
    Updating `C:\Users\nickb\Documents\vit-gradient-flow\Project.toml`
  [872c559c] + NNlib v0.9.33
    Manifest No packages added to or removed from `C:\Users\nickb\Documents\vit-gradient-flow\Manifest.toml`


Environment activated and libraries loaded ✓


## 2. Restoring the Trained Models
We instantiate the Xavier and Orthogonal models and inject the trained parameters (`model_state`) that we saved to our hard drive. We also load the MNIST test dataset to evaluate their predictions.

In [4]:


# 1. Initializers
function xavier_init(dims...)
    return Flux.glorot_uniform(dims...)
end

function orthogonal_init(dims...)
    return Flux.orthogonal(dims...)
end

# 2. Patch Embedding Layer
struct PatchEmbedding{P, C, E}
    proj::P
    cls_token::C
    pos_embed::E
end

Flux.@functor PatchEmbedding

function PatchEmbedding(img_size::Int, patch_size::Int, in_channels::Int, embed_dim::Int, init)
    num_patches = (img_size ÷ patch_size)^2
    proj = Conv((patch_size, patch_size), in_channels => embed_dim, stride=patch_size, init=init)
    cls_token = init(embed_dim, 1, 1)
    pos_embed = init(embed_dim, num_patches + 1, 1)
    return PatchEmbedding(proj, cls_token, pos_embed)
end

function (m::PatchEmbedding)(x::AbstractArray)
    # Extract patches: (H, W, C, B) -> (embed_dim, num_patches, B)
    x = m.proj(x)
    x = reshape(x, size(x, 1), :, size(x, 4))
    
    # Expand cls_token for the batch and concatenate
    batch_size = size(x, 3)
    cls_tokens = repeat(m.cls_token, 1, 1, batch_size)
    x = cat(cls_tokens, x, dims=2)
    
    # Add positional embedding
    return x .+ m.pos_embed
end

# 3. Transformer Block
struct TransformerBlock{L1, A, L2, F}
    ln1::L1
    attn::A
    ln2::L2
    ff::F
end

Flux.@functor TransformerBlock

function TransformerBlock(embed_dim::Int, num_heads::Int, mlp_ratio::Int, init)
    ln1 = LayerNorm(embed_dim)
    attn = MultiHeadAttention(embed_dim, num_heads) # Assuming standard Flux MHA
    ln2 = LayerNorm(embed_dim)
    ff = Chain(
        Dense(embed_dim, embed_dim * mlp_ratio, gelu, init=init),
        Dense(embed_dim * mlp_ratio, embed_dim, init=init)
    )
    return TransformerBlock(ln1, attn, ln2, ff)
end

function (m::TransformerBlock)(x::AbstractArray)
    x = x .+ m.attn(m.ln1(x), m.ln1(x), m.ln1(x))[1]
    x = x .+ m.ff(m.ln2(x))
    return x
end

# 4. Full Vision Transformer
struct ViT{P, B, L, H}
    patch_embed::P
    blocks::B
    ln::L
    head::H
end

Flux.@functor ViT

function ViT(init)
    patch_embed = PatchEmbedding(IMG_SIZE, PATCH_SIZE, IN_CHANNELS, EMBED_DIM, init)
    blocks = [TransformerBlock(EMBED_DIM, NUM_HEADS, MLP_RATIO, init) for _ in 1:NUM_BLOCKS]
    ln = LayerNorm(EMBED_DIM)
    head = Dense(EMBED_DIM, NUM_CLASSES, init=init)
    return ViT(patch_embed, blocks, ln, head)
end

function (m::ViT)(x::AbstractArray)
    x = m.patch_embed(x)
    for block in m.blocks
        x = block(x)
    end
    x = m.ln(x)
    # Extract only the CLS token (first token) for classification
    cls_out = x[:, 1, :]
    return m.head(cls_out)
end

println("ViT Architecture defined ✓")

ViT Architecture defined ✓


┌ Warning: The use of `Flux.@functor` is deprecated.
│ Most likely, you should write `Flux.@layer MyLayer`which will add various convenience methods for your type,such as pretty-printing and use with Adapt.jl.
│ However, this is not required. Flux.jl v0.15 uses Functors.jl v0.5,which makes exploration of most nested `struct`s opt-out instead of opt-in...so Flux will automatically see inside any custom struct definitions.
│ If you really want to apply the `@functor` macro to a custom struct, use `Functors.@functor` instead.
└ @ Flux C:\Users\nickb\.julia\packages\Flux\DZYiO\src\deprecations.jl:101


UndefVarError: UndefVarError: `model_xavier` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.